# Retrain the two Bayesian factorization machines (CPU, one core)

**Run All** together with `retrain_tuned_models.ipynb`, which blends Task 3 / Task 1 as soon as these models are stored.

- Task 3: time-aware Bayesian FM - rank 64, weekday + month + year, 30 + 380 draws (sweep: 0.84445) - ~35 min per fit, twice (refit)
- Task 1: Bayesian FM - rank 128, 20 + 180 draws (sweep: 0.72022) - ~35 min per fit, twice (refit)

Log: `trained_models/retrain_bayesian_fm_log.txt`.

In [ ]:
# no autoreload: the run uses the code and lib/config.py as they are now
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path
sys.path.append(str(Path.cwd()))               # assignment1.py and lib/ live next to this notebook (in submission/)

import numpy as np
import torch

import assignment1
import lib.config as model_configs
from lib.config import TRAINED_MODELS_DIR
from lib.helper_functioins import mse
from lib.model_storage import load_model_result, save_model_result, trained_model_path
from lib.models.blender import store_blender
from lib.models.pair_blender import store_pair_blender

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

LOG_PATH = TRAINED_MODELS_DIR / "retrain_bayesian_fm_log.txt"
run_start = time.time()                         # stored files newer than this were written by this run


def write_log(line: str):
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} {line}\n")
    print(line, flush=True)


# keeps Windows from sleeping while the notebook runs (released in the last cell)
def keep_computer_awake(enabled: bool):
    if sys.platform == "win32":
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001 if enabled else 0x80000000)


# trains one sub-model and stores it under its model name in trained_models/<task>/; a failure is logged and skipped
# (the blend then uses the version stored before, if there is one)
def train_and_store(task_name: str, task, model_name: str, trainer) -> bool:
    write_log(f"start    {task_name} {model_name}")
    start = time.perf_counter()
    try:
        result = trainer(task)
    except Exception as error:
        traceback.print_exc()
        write_log(f"FAILED   {task_name} {model_name}: {type(error).__name__}: {error}"[:400])
        return False
    results = result if isinstance(result, dict) else {model_name: result}
    for name, model_result in results.items():
        save_model_result(model_result, task_name, name)
        score = (f" | valid MSE {mse(model_result.valid_predictions, task.validation_data.ratings):.5f}"
                 if hasattr(task, "validation_data") and "zero_model" not in name else "")   # zero models predict P(rating 0)
        write_log(f"stored   {task_name} {name} ({(time.perf_counter() - start) / 60:.1f} min){score}")
    torch.cuda.empty_cache()
    return True


keep_computer_awake(True)
write_log(f"run started ({Path.cwd().name})")

In [ ]:
task3 = assignment1.load_task3()
train_and_store("task3", task3, "time_bayesian_factorization_machine", assignment1.train_task3_time_bayesian_factorization_machine)
del task3

In [ ]:
task1 = assignment1.load_task1()
train_and_store("task1", task1, "bayesian_factorization_machine", assignment1.train_task1_bayesian_factorization_machine)
del task1

In [ ]:
keep_computer_awake(False)
write_log("run finished")